# FINAL_6 — Grad-CAM + XAlign: Analisis TP / TN / FP / FN
## Brain Tumor Classification — EfficientNetB0 + Explainable AI

## 1. Mount Google Drive & Import Library

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
import re
import json
import cv2
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.spatial.distance import directed_hausdorff

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)


## 2. Konfigurasi Path

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/Tugas Akhir/TL Gradcam/TransferLearning_KEDUA_GC_test'
PROSES_DIR = '/content/drive/MyDrive/Tugas Akhir/TL Gradcam/TransferLearning_KEDUA_GC_test/proses'

TEST_DIR   = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_GC_test/test'
VAL_DIR    = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_GC_test/val'

PAIR_BASE_DIR      = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_GC_test'
PAIR_TEST_DIR      = os.path.join(PAIR_BASE_DIR, 'test',  'tumor')
PAIR_VAL_DIR       = os.path.join(PAIR_BASE_DIR, 'val',   'tumor')
PAIR_TEST_MASK_DIR = os.path.join(PAIR_TEST_DIR, 'masks')
PAIR_VAL_MASK_DIR  = os.path.join(PAIR_VAL_DIR,  'masks')

# Pola nama file hasil pairing image-mask (dari notebook 7)
IMAGE_PATTERN = r'^image_tumor_mask__(.+)_CT_s(.+)\.png$'
MASK_TEMPLATE = 'mask_tumor__{patient_id}_CT_m{slice_id}.png'

XAI_DIR      = os.path.join(PROSES_DIR, 'gradcam_xai')
XALIGN_DIR   = os.path.join(XAI_DIR, 'xalign')
QUAD_DIR     = os.path.join(XAI_DIR, 'quadrant_analysis')

for d in [XAI_DIR, XALIGN_DIR, QUAD_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE    = (224, 224)
CLASS_NAMES = ['no_tumor', 'tumor']
NUM_CLASSES = len(CLASS_NAMES)

XALIGN_ALPHA = 0.5
XALIGN_BETA  = 0.4
XALIGN_GAMMA = 0.1

# Peringatan jika path penting tidak ditemukan
for p, name in [
    (PROSES_DIR,          'PROSES_DIR'),
    (TEST_DIR,            'TEST_DIR'),
    (PAIR_TEST_DIR,       'PAIR_TEST_DIR'),
    (PAIR_TEST_MASK_DIR,  'PAIR_TEST_MASK_DIR'),
]:
    if not os.path.exists(p):
        print(f'PERINGATAN: {name} tidak ditemukan -> {p}')

## 3. Load Model Terbaik dari FINAL_4_c

In [ ]:
config_path = os.path.join(PROSES_DIR, 'best_config_tl.json')
assert os.path.exists(config_path), f'File tidak ditemukan: {config_path}'
with open(config_path, 'r') as f:
    best_config = json.load(f)

model_path = os.path.join(PROSES_DIR, 'best_model_tl_efficientnet.keras')
assert os.path.exists(model_path), f'File tidak ditemukan: {model_path}'
model = keras.models.load_model(model_path)
print(f'Model dimuat: {model_path}')

## 4. Deteksi Otomatis Last Conv Layer

In [ ]:
BACKBONE_NAME   = None
LAST_CONV_LAYER = None

for layer in model.layers:
    if hasattr(layer, 'layers'):
        BACKBONE_NAME = layer.name
        for sub in layer.layers:
            if isinstance(sub, tf.keras.layers.Conv2D):
                LAST_CONV_LAYER = sub.name  # terus update -> ambil conv terakhir

print(f'Backbone: {BACKBONE_NAME}  |  Last Conv2D: {LAST_CONV_LAYER}')

## 5. Implementasi Grad-CAM

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name,
                          backbone_layer_name=None, pred_index=None):
    # Dual-output model (conv_feature_map, prediksi) agar gradient path selalu terdefinisi
    if backbone_layer_name is not None:
        backbone = model.get_layer(backbone_layer_name)
    else:
        backbone = next(l for l in model.layers if hasattr(l, 'layers'))

    target_conv  = backbone.get_layer(last_conv_layer_name)
    conv_out     = target_conv.output
    head_input   = backbone.output
    x            = head_input
    after_bb     = False
    for layer in model.layers:
        if layer.name == backbone.name:
            after_bb = True
            continue
        if after_bb:
            x = layer(x, training=False)
    final_output = x

    last_layer = model.layers[-1]
    last_layer.activation = None  # nonaktifkan softmax sementara agar diferensiasi logit

    grad_model = keras.Model(
        inputs  = backbone.input,
        outputs = [conv_out, final_output]
    )

    img_tensor = tf.cast(img_array, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_outputs, logits = grad_model(img_tensor, training=False)
        if pred_index is None:
            pred_index = int(tf.argmax(logits[0]).numpy())
        class_channel = logits[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    last_layer.activation = tf.keras.activations.softmax
    if grads is None:
        raise RuntimeError('Gradien None - periksa arsitektur model.')

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_grads, axis=-1).numpy()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    all_probs = tf.nn.softmax(logits)[0].numpy()
    pred_prob = float(all_probs[pred_index])
    return heatmap, int(pred_index), pred_prob, all_probs


def superimpose_heatmap(heatmap, original_img_rgb, alpha=0.5, colormap=cv2.COLORMAP_JET):
    h, w = original_img_rgb.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_uint8   = np.uint8(255 * heatmap_resized)
    colored = cv2.applyColorMap(heatmap_uint8, colormap)
    colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(
        original_img_rgb.astype(np.float32), 1 - alpha,
        colored.astype(np.float32), alpha, 0
    )
    return np.uint8(superimposed)


def gradcam_single(img_rgb, model, last_conv, backbone_name,
                    target_class=None, alpha=0.5):
    img_arr = np.expand_dims(img_rgb.astype(np.float32), 0)
    heatmap, pred_idx, pred_prob, all_probs = make_gradcam_heatmap(
        img_arr, model, last_conv,
        backbone_layer_name=backbone_name,
        pred_index=target_class
    )
    heatmap_224  = cv2.resize(heatmap, IMG_SIZE)
    superimposed = superimpose_heatmap(heatmap_224, img_rgb, alpha=alpha)
    return heatmap_224, superimposed, pred_idx, pred_prob, all_probs


def format_class_probs(all_probs, class_names=None):
    names = class_names if class_names is not None else CLASS_NAMES
    return '  |  '.join(f'{n}: {p:.1%}' for n, p in zip(names, all_probs))


# Uji cepat dengan dummy input, hentikan notebook jika Grad-CAM gagal
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
try:
    make_gradcam_heatmap(dummy, model, LAST_CONV_LAYER, BACKBONE_NAME)
except Exception as e:
    print(f'ERROR fatal Grad-CAM: {e}')
    raise

## 6. Implementasi XAlign (Muhammad & Bendechache 2025)

In [ ]:
def compute_wro(saliency_map: np.ndarray, gt_mask: np.ndarray) -> float:
    # WRO: proporsi relevansi saliency di dalam ground-truth mask (Eq. 18)
    total = saliency_map.sum()
    if total < 1e-8:
        return 0.0
    return float((saliency_map * gt_mask).sum() / total)


def compute_bas(saliency_map: np.ndarray, gt_mask: np.ndarray,
                threshold: float = 0.5) -> float:
    # BAS: inverse Hausdorff Distance antara batas saliency dan mask (Eq. 19)
    H, W = saliency_map.shape
    sal_binary = (saliency_map >= threshold).astype(np.uint8)
    gt_binary  = gt_mask.astype(np.uint8)

    contours_sal, _ = cv2.findContours(sal_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    contours_gt,  _ = cv2.findContours(gt_binary,  cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    if len(contours_sal) == 0 or len(contours_gt) == 0:
        return 0.0

    pts_sal = np.vstack([c.reshape(-1, 2) for c in contours_sal]).astype(float)
    pts_gt  = np.vstack([c.reshape(-1, 2) for c in contours_gt ]).astype(float)

    hd = max(directed_hausdorff(pts_sal, pts_gt)[0],
             directed_hausdorff(pts_gt, pts_sal)[0])
    return float(np.clip(1.0 - hd / max(H, W), 0.0, 1.0))


def compute_dp(saliency_map: np.ndarray, gt_mask: np.ndarray) -> float:
    # DP: proporsi relevansi di LUAR mask, penalti dispersi (Eq. 20)
    total = saliency_map.sum()
    if total < 1e-8:
        return 0.0
    return float((saliency_map * (1.0 - gt_mask)).sum() / total)


def compute_xalign(saliency_map, gt_mask,
                    alpha=0.5, beta=0.4, gamma=0.1, threshold=0.5):
    # XAlign = alpha*WRO + beta*BAS - gamma*DP (Eq. 17)
    s = saliency_map.astype(np.float32)
    if s.max() > 1.0 + 1e-6:
        s /= s.max()

    g = gt_mask.astype(np.float32)
    if s.shape != g.shape:
        g = cv2.resize(g, (s.shape[1], s.shape[0]), interpolation=cv2.INTER_NEAREST)
    g = (g > 0.5).astype(np.float32)

    wro = compute_wro(s, g)
    bas = compute_bas(s, g, threshold=threshold)
    dp  = compute_dp(s, g)
    xalign_score = float(np.clip(alpha * wro + beta * bas - gamma * dp, 0.0, 1.0))

    return {'wro': wro, 'bas': bas, 'dp': dp, 'xalign': xalign_score}

## 7. Load Test Images

In [ ]:
def load_test_images(test_dir, class_names, img_size=(224, 224), max_per_class=100):
    imgs, labels, paths = [], [], []
    for cls_idx, cls_name in enumerate(class_names):
        cls_dir = os.path.join(test_dir, cls_name)
        if not os.path.isdir(cls_dir):
            print(f'PERINGATAN: folder tidak ditemukan, dilewati -> {cls_dir}')
            continue
        img_paths = sorted(
            p for ext in ('*.png', '*.jpg', '*.jpeg')
            for p in glob.glob(os.path.join(cls_dir, ext))
        )[:max_per_class]
        for fp in img_paths:
            raw = cv2.imread(fp)
            if raw is None:
                continue
            img_rgb = cv2.cvtColor(cv2.resize(raw, img_size), cv2.COLOR_BGR2RGB)
            imgs.append(img_rgb)
            labels.append(cls_idx)
            paths.append(fp)
    return imgs, labels, paths


test_imgs, test_labels, test_paths = load_test_images(
    TEST_DIR, CLASS_NAMES, IMG_SIZE, max_per_class=100
)
print(f'Total test images: {len(test_imgs)}')

## 8. Prediksi Model pada Seluruh Test Images

In [ ]:
X_test_arr = np.array(test_imgs, dtype=np.float32)

BATCH = 32
preds_prob = []
for i in range(0, len(X_test_arr), BATCH):
    batch = X_test_arr[i:i+BATCH]
    preds_prob.append(model.predict(batch, verbose=0))
preds_prob = np.concatenate(preds_prob, axis=0)
y_pred     = np.argmax(preds_prob, axis=1)
y_true     = np.array(test_labels)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Kategorisasi kuadran confusion matrix (kelas positif = tumor/index 1)
tp_idx, tn_idx, fp_idx, fn_idx = [], [], [], []
for i, (yt, yp) in enumerate(zip(y_true, y_pred)):
    if   yt == 1 and yp == 1: tp_idx.append(i)
    elif yt == 0 and yp == 0: tn_idx.append(i)
    elif yt == 0 and yp == 1: fp_idx.append(i)
    elif yt == 1 and yp == 0: fn_idx.append(i)

quadrant_idx = {'TP': tp_idx, 'TN': tn_idx, 'FP': fp_idx, 'FN': fn_idx}

print(f'TP={len(tp_idx)}  TN={len(tn_idx)}  FP={len(fp_idx)}  FN={len(fn_idx)}  Total={len(y_true)}')

## 9. Confusion Matrix dengan Anotasi TP/TN/FP/FN

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
            linewidths=1.5, linecolor='white')

# Anotasi manual label TP/TN/FP/FN per sel
labels_quad = [['TN', 'FP'], ['FN', 'TP']]
colors_quad = [['#27AE60', '#E74C3C'], ['#E74C3C', '#27AE60']]

for row_i in range(2):
    for col_i in range(2):
        count = cm[row_i, col_i]
        quad  = labels_quad[row_i][col_i]
        color = colors_quad[row_i][col_i]
        ax.text(col_i + 0.5, row_i + 0.35, str(count),
                ha='center', va='center', fontsize=22, fontweight='bold', color='white')
        ax.text(col_i + 0.5, row_i + 0.65, quad,
                ha='center', va='center', fontsize=13, fontweight='bold',
                color=color,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor=color))

ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Confusion Matrix — EfficientNetB0 + XAI\n(TP / TN / FP / FN)',
             fontsize=14, fontweight='bold')

patch_tp = mpatches.Patch(color='#27AE60', label='TP / TN = benar')
patch_fp = mpatches.Patch(color='#E74C3C', label='FP / FN = salah')
ax.legend(handles=[patch_tp, patch_fp], loc='upper right', fontsize=10)

plt.tight_layout()
out_path = os.path.join(QUAD_DIR, 'confusion_matrix_TP_TN_FP_FN.png')
plt.savefig(out_path, dpi=150)
plt.show()

## 10. Load Ground-Truth Mask via Pair Table

In [ ]:
def build_pair_table(image_dir, mask_dir, split,
                     image_pattern=None, mask_template=None):
    from pathlib import Path as _Path
    _pattern  = image_pattern if image_pattern is not None else IMAGE_PATTERN
    _template = mask_template if mask_template is not None else MASK_TEMPLATE
    image_dir = _Path(image_dir)
    mask_dir  = _Path(mask_dir)
    records = []
    for img_path in sorted(image_dir.glob('*.png')):
        m = re.match(_pattern, img_path.name)
        if m is None:
            continue
        patient_id = m.group(1)
        slice_id   = m.group(2)
        mask_name  = _template.format(patient_id=patient_id, slice_id=slice_id)
        msk_path   = mask_dir / mask_name
        records.append({
            'split': split, 'image_name': img_path.name,
            'mask_name': mask_name,
            'image_path': str(img_path), 'mask_path': str(msk_path),
            'mask_exists': msk_path.is_file(),
        })
    return pd.DataFrame(records)


def load_gt_mask(img_path, pair_lookup, img_size=(224, 224)):
    # Load ground-truth mask dari pair lookup, None jika tidak ada
    img_name = os.path.basename(img_path)
    if img_name in pair_lookup:
        mask_path = pair_lookup[img_name]
        if os.path.exists(mask_path):
            raw = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if raw is not None:
                resized = cv2.resize(raw, img_size, interpolation=cv2.INTER_NEAREST)
                return (resized > 127).astype(np.float32)
    return None


PAIR_AVAILABLE = os.path.exists(PAIR_TEST_DIR) and os.path.exists(PAIR_TEST_MASK_DIR)

if PAIR_AVAILABLE:
    df_pair_test = build_pair_table(PAIR_TEST_DIR, PAIR_TEST_MASK_DIR, 'test')
    df_pair_val  = build_pair_table(PAIR_VAL_DIR,  PAIR_VAL_MASK_DIR,  'val')
    df_pair_all  = pd.concat([df_pair_test, df_pair_val], ignore_index=True)
    pair_lookup  = {row['image_name']: row['mask_path']
                    for _, row in df_pair_all.iterrows() if row['mask_exists']}
    MASKS_AVAILABLE = len(pair_lookup) > 0
    df_pair_all.to_csv(os.path.join(QUAD_DIR, 'pair_table.csv'), index=False)
else:
    pair_lookup = {}
    MASKS_AVAILABLE = False
    print(f'PERINGATAN: direktori mask tidak ditemukan ({PAIR_TEST_DIR}). XAlign akan dilewati.')

test_masks = []
n_found    = 0
for fp in test_paths:
    mask = load_gt_mask(fp, pair_lookup, IMG_SIZE)
    test_masks.append(mask)
    if mask is not None:
        n_found += 1

print(f'Mask berhasil dimuat: {n_found} / {len(test_paths)}')
if n_found == 0 and PAIR_AVAILABLE:
    print('PERINGATAN: tidak ada mask yang cocok. Periksa IMAGE_PATTERN dan MASK_TEMPLATE.')

## 11. Fungsi Visualisasi 4-Panel per Gambar

In [ ]:
def make_mask_vis(img_rgb, mask):
    # Overlay kuning + kontur mask di atas gambar asli
    vis = img_rgb.copy()
    if mask is not None and mask.sum() > 0:
        mask_bool = mask.astype(bool)
        overlay   = vis.copy()
        overlay[mask_bool] = [255, 230, 0]
        vis = cv2.addWeighted(vis.astype(np.float32), 0.5,
                               overlay.astype(np.float32), 0.5, 0).astype(np.uint8)
        contours, _ = cv2.findContours(mask.astype(np.uint8),
                                        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, contours, -1, (255, 200, 0), 2)
    else:
        vis = img_rgb.copy()
    return vis


def plot_quadrant_grid(
    indices, quadrant_name, quadrant_color,
    test_imgs, test_labels, test_paths, test_masks,
    model, last_conv, backbone_name,
    n_show=6, alpha=0.5, save_dir=None
):
    # Grid 4-panel (Original | Heatmap | Mask | Overlay) untuk n_show gambar dari satu kuadran
    if len(indices) == 0:
        print(f'PERINGATAN: tidak ada gambar untuk kuadran {quadrant_name}.')
        return

    n = min(n_show, len(indices))
    selected = indices[:n]

    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['Original CT Scan', 'Grad-CAM Heatmap',
                  'Ground-Truth Mask', 'Overlay (GradCAM)']
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=13, fontweight='bold', pad=8)

    for row_i, idx in enumerate(selected):
        img_rgb   = test_imgs[idx]
        true_lbl  = test_labels[idx]
        mask      = test_masks[idx]
        fp        = test_paths[idx]

        heatmap, overlay, pred_idx, pred_prob, all_probs = gradcam_single(
            img_rgb, model, last_conv, backbone_name, alpha=alpha
        )
        true_name = CLASS_NAMES[true_lbl]
        pred_name = CLASS_NAMES[pred_idx]
        probs_str = format_class_probs(all_probs)

        # XAlign hanya dihitung untuk kelas tumor dengan mask tersedia
        xalign_str = ''
        if mask is not None and true_lbl == 1:
            sc = compute_xalign(heatmap, mask, XALIGN_ALPHA, XALIGN_BETA, XALIGN_GAMMA)
            xalign_str = (f'XAlign={sc["xalign"]:.3f}  '
                          f'WRO={sc["wro"]:.3f}  '
                          f'BAS={sc["bas"]:.3f}  '
                          f'DP={sc["dp"]:.3f}')

        row_label = (f'True: {true_name}\nPred: {pred_name} ({pred_prob:.1%})\n{probs_str}')
        if xalign_str:
            row_label += f'\n{xalign_str}'
        axes[row_i, 0].set_ylabel(row_label, fontsize=9,
                                   rotation=0, labelpad=180, va='center')

        axes[row_i, 0].imshow(img_rgb)
        axes[row_i, 0].axis('off')

        im = axes[row_i, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
        plt.colorbar(im, ax=axes[row_i, 1], fraction=0.046, pad=0.04)
        axes[row_i, 1].axis('off')

        mask_vis = make_mask_vis(img_rgb, mask)
        axes[row_i, 2].imshow(mask_vis)
        if mask is None or mask.sum() == 0:
            axes[row_i, 2].set_xlabel('Mask tidak tersedia / kosong',
                                       fontsize=8, color='gray')
        axes[row_i, 2].axis('off')

        axes[row_i, 3].imshow(overlay)
        axes[row_i, 3].axis('off')

    fig.suptitle(
        f'Kuadran {quadrant_name}\n'
        f'({len(indices)} total gambar — menampilkan {n})',
        fontsize=15, fontweight='bold', color=quadrant_color, y=1.01
    )
    plt.tight_layout()

    if save_dir:
        out_path = os.path.join(save_dir, f'quadrant_{quadrant_name}.png')
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

## 12. Visualisasi Kuadran TP (True Positive)

In [ ]:
print(f'Kuadran TP: {len(tp_idx)} gambar')
plot_quadrant_grid(
    indices       = tp_idx,
    quadrant_name = 'TP',
    quadrant_color= '#1A7F37',
    test_imgs=test_imgs, test_labels=test_labels,
    test_paths=test_paths, test_masks=test_masks,
    model=model, last_conv=LAST_CONV_LAYER, backbone_name=BACKBONE_NAME,
    n_show=6, alpha=0.5, save_dir=QUAD_DIR
)

## 13. Visualisasi Kuadran TN (True Negative)

In [ ]:
print(f'Kuadran TN: {len(tn_idx)} gambar')
plot_quadrant_grid(
    indices       = tn_idx,
    quadrant_name = 'TN',
    quadrant_color= '#1A7F37',
    test_imgs=test_imgs, test_labels=test_labels,
    test_paths=test_paths, test_masks=test_masks,
    model=model, last_conv=LAST_CONV_LAYER, backbone_name=BACKBONE_NAME,
    n_show=6, alpha=0.5, save_dir=QUAD_DIR
)

## 14. Visualisasi Kuadran FP (False Positive)

In [ ]:
print(f'Kuadran FP: {len(fp_idx)} gambar')
if len(fp_idx) == 0:
    print('Tidak ada False Positive.')
else:
    plot_quadrant_grid(
        indices       = fp_idx,
        quadrant_name = 'FP',
        quadrant_color= '#C0392B',
        test_imgs=test_imgs, test_labels=test_labels,
        test_paths=test_paths, test_masks=test_masks,
        model=model, last_conv=LAST_CONV_LAYER, backbone_name=BACKBONE_NAME,
        n_show=6, alpha=0.5, save_dir=QUAD_DIR
    )

## 15. Visualisasi Kuadran FN (False Negative)

In [ ]:
print(f'Kuadran FN: {len(fn_idx)} gambar')
if len(fn_idx) == 0:
    print('Tidak ada False Negative.')
else:
    plot_quadrant_grid(
        indices       = fn_idx,
        quadrant_name = 'FN',
        quadrant_color= '#C0392B',
        test_imgs=test_imgs, test_labels=test_labels,
        test_paths=test_paths, test_masks=test_masks,
        model=model, last_conv=LAST_CONV_LAYER, backbone_name=BACKBONE_NAME,
        n_show=6, alpha=0.5, save_dir=QUAD_DIR
    )

## 16. Komputasi Grad-CAM + XAlign untuk SEMUA Gambar

In [ ]:
all_results = []

for i, (img_rgb, lbl, fp, mask) in enumerate(
        zip(test_imgs, test_labels, test_paths, test_masks)):

    img_arr = np.expand_dims(img_rgb.astype(np.float32), 0)
    heatmap, pred_idx, pred_prob, all_probs = make_gradcam_heatmap(
        img_arr, model, LAST_CONV_LAYER, BACKBONE_NAME
    )
    heatmap_224 = cv2.resize(heatmap, IMG_SIZE)

    yt, yp = lbl, pred_idx
    if   yt == 1 and yp == 1: quad = 'TP'
    elif yt == 0 and yp == 0: quad = 'TN'
    elif yt == 0 and yp == 1: quad = 'FP'
    else:                     quad = 'FN'

    rec = {
        'img_path'  : fp,
        'img_name'  : os.path.basename(fp),
        'true_label': lbl,
        'pred_label': pred_idx,
        'true_name' : CLASS_NAMES[lbl],
        'pred_name' : CLASS_NAMES[pred_idx],
        'pred_prob' : pred_prob,
        'quadrant'  : quad,
    }
    for cls_i, cls_n in enumerate(CLASS_NAMES):
        rec[f'prob_{cls_n}'] = float(all_probs[cls_i])

    # XAlign hanya untuk gambar tumor dengan mask tersedia
    if mask is not None and lbl == 1:
        sc = compute_xalign(heatmap_224, mask, XALIGN_ALPHA, XALIGN_BETA, XALIGN_GAMMA)
        rec.update(sc)
    else:
        rec.update({'wro': None, 'bas': None, 'dp': None, 'xalign': None})

    all_results.append(rec)

df_all = pd.DataFrame(all_results)
csv_path = os.path.join(QUAD_DIR, 'all_results_with_xalign.csv')
df_all.to_csv(csv_path, index=False)
print(f'Hasil lengkap disimpan: {csv_path}')
print(df_all['quadrant'].value_counts().to_string())

## 17. Statistik XAlign per Kuadran (TP / FN)

## 17. Statistik & Bar Chart XAlign per Kuadran

In [ ]:
df_with_xa = df_all.dropna(subset=['xalign'])

if len(df_with_xa) == 0:
    print('PERINGATAN: tidak ada data XAlign. Periksa ketersediaan mask.')
else:
    stats = df_with_xa.groupby('quadrant')[['xalign', 'wro', 'bas', 'dp']].agg(
        ['mean', 'std', 'count']
    ).round(4)
    print(stats)

    metrics_plot = ['xalign', 'wro', 'bas', 'dp']
    quads_avail  = df_with_xa['quadrant'].unique()
    colors_quad  = {'TP': '#27AE60', 'FN': '#C0392B', 'FP': '#E67E22', 'TN': '#2980B9'}

### 17a. Bar Chart — XAlign

In [ ]:
if len(df_with_xa) > 0:
    metric = 'xalign'
    means  = [df_with_xa[df_with_xa['quadrant'] == q][metric].mean() for q in quads_avail]
    stds   = [df_with_xa[df_with_xa['quadrant'] == q][metric].std()  for q in quads_avail]
    colors = [colors_quad.get(q, 'gray') for q in quads_avail]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(quads_avail, means, yerr=stds, color=colors,
           edgecolor='black', capsize=6, linewidth=0.8)
    for j, (m, s) in enumerate(zip(means, stds)):
        ax.text(j, m + (s if not np.isnan(s) else 0) + 0.02,
                f'{m:.3f}', ha='center', va='bottom', fontsize=11)
    ax.set_title('XAlign per Kuadran', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.25)
    ax.axhline(y=1.0, linestyle='--', color='gray', alpha=0.4)
    ax.set_xlabel('Kuadran'); ax.set_ylabel('Score')
    plt.tight_layout()
    out_path = os.path.join(XALIGN_DIR, 'bar_xalign.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

### 17b. Bar Chart — WRO

In [ ]:
if len(df_with_xa) > 0:
    metric = 'wro'
    means  = [df_with_xa[df_with_xa['quadrant'] == q][metric].mean() for q in quads_avail]
    stds   = [df_with_xa[df_with_xa['quadrant'] == q][metric].std()  for q in quads_avail]
    colors = [colors_quad.get(q, 'gray') for q in quads_avail]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(quads_avail, means, yerr=stds, color=colors,
           edgecolor='black', capsize=6, linewidth=0.8)
    for j, (m, s) in enumerate(zip(means, stds)):
        ax.text(j, m + (s if not np.isnan(s) else 0) + 0.02,
                f'{m:.3f}', ha='center', va='bottom', fontsize=11)
    ax.set_title('WRO (Weighted Relevance Overlap) per Kuadran', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.25)
    ax.axhline(y=1.0, linestyle='--', color='gray', alpha=0.4)
    ax.set_xlabel('Kuadran'); ax.set_ylabel('Score')
    plt.tight_layout()
    out_path = os.path.join(XALIGN_DIR, 'bar_wro.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

### 17c. Bar Chart — BAS

In [ ]:
if len(df_with_xa) > 0:
    metric = 'bas'
    means  = [df_with_xa[df_with_xa['quadrant'] == q][metric].mean() for q in quads_avail]
    stds   = [df_with_xa[df_with_xa['quadrant'] == q][metric].std()  for q in quads_avail]
    colors = [colors_quad.get(q, 'gray') for q in quads_avail]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(quads_avail, means, yerr=stds, color=colors,
           edgecolor='black', capsize=6, linewidth=0.8)
    for j, (m, s) in enumerate(zip(means, stds)):
        ax.text(j, m + (s if not np.isnan(s) else 0) + 0.02,
                f'{m:.3f}', ha='center', va='bottom', fontsize=11)
    ax.set_title('BAS (Boundary Agreement Score) per Kuadran', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.25)
    ax.axhline(y=1.0, linestyle='--', color='gray', alpha=0.4)
    ax.set_xlabel('Kuadran'); ax.set_ylabel('Score')
    plt.tight_layout()
    out_path = os.path.join(XALIGN_DIR, 'bar_bas.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

### 17d. Bar Chart — DP

In [ ]:
if len(df_with_xa) > 0:
    metric = 'dp'
    means  = [df_with_xa[df_with_xa['quadrant'] == q][metric].mean() for q in quads_avail]
    stds   = [df_with_xa[df_with_xa['quadrant'] == q][metric].std()  for q in quads_avail]
    colors = [colors_quad.get(q, 'gray') for q in quads_avail]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(quads_avail, means, yerr=stds, color=colors,
           edgecolor='black', capsize=6, linewidth=0.8)
    for j, (m, s) in enumerate(zip(means, stds)):
        ax.text(j, m + (s if not np.isnan(s) else 0) + 0.02,
                f'{m:.3f}', ha='center', va='bottom', fontsize=11)
    ax.set_title('DP (Dispersion Penalty) per Kuadran', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.25)
    ax.axhline(y=1.0, linestyle='--', color='gray', alpha=0.4)
    ax.set_xlabel('Kuadran'); ax.set_ylabel('Score')
    plt.tight_layout()
    out_path = os.path.join(XALIGN_DIR, 'bar_dp.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

## 18. Visualisasi XAlign — Contoh Terbaik & Terburuk per Kuadran

In [ ]:
def plot_xalign_topN(df_results, all_test_imgs, all_test_paths, all_test_masks,
                     model, last_conv, backbone_name,
                     quadrant_filter=None, sort_by='xalign',
                     ascending=False, n=4, save_path=None):
    # Visualisasi n gambar terbaik/terburuk XAlign, opsional filter kuadran
    df = df_results.dropna(subset=['xalign'])
    if quadrant_filter:
        df = df[df['quadrant'] == quadrant_filter]
    df = df.sort_values(sort_by, ascending=ascending).head(n)

    if len(df) == 0:
        print(f'PERINGATAN: tidak ada data XAlign untuk kuadran {quadrant_filter}.')
        return

    fig, axes = plt.subplots(len(df), 4, figsize=(20, 5 * len(df)))
    if len(df) == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['Original CT Scan', 'Grad-CAM Heatmap',
                  'Ground-Truth Mask', 'Overlay + Skor']
    for ax, t in zip(axes[0], col_titles):
        ax.set_title(t, fontsize=12, fontweight='bold')

    for row_i, (_, row) in enumerate(df.iterrows()):
        fp = row['img_path']
        try:
            gi = next(i for i, p in enumerate(all_test_paths) if p == fp)
        except StopIteration:
            continue

        img_rgb = all_test_imgs[gi]
        mask    = all_test_masks[gi]

        heatmap, overlay, pred_idx, pred_prob, all_probs = gradcam_single(
            img_rgb, model, last_conv, backbone_name, alpha=0.5
        )
        mask_vis = make_mask_vis(img_rgb, mask)
        probs_str = format_class_probs(all_probs)

        row_label = (
            f'True: {row["true_name"]}\nPred: {row["pred_name"]} ({pred_prob:.1%})\n'
            f'{probs_str}\n'
            f'[{row["quadrant"]}] XAlign={row["xalign"]:.4f}\n'
            f'WRO={row["wro"]:.3f}  BAS={row["bas"]:.3f}  DP={row["dp"]:.3f}'
        )
        axes[row_i, 0].set_ylabel(row_label, fontsize=8.5, rotation=0,
                                   labelpad=185, va='center')

        axes[row_i, 0].imshow(img_rgb);               axes[row_i, 0].axis('off')
        im = axes[row_i, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
        plt.colorbar(im, ax=axes[row_i, 1], fraction=0.046); axes[row_i, 1].axis('off')
        axes[row_i, 2].imshow(mask_vis);               axes[row_i, 2].axis('off')
        axes[row_i, 3].imshow(overlay);                axes[row_i, 3].axis('off')

        color = '#1A7F37' if row['quadrant'] in ('TP', 'TN') else '#C0392B'
        axes[row_i, 3].set_title(
            f'{row["quadrant"]}  XAlign = {row["xalign"]:.4f}',
            fontsize=10, color=color
        )

    direction = 'Tertinggi' if not ascending else 'Terendah'
    quad_str  = f'Kuadran {quadrant_filter} — ' if quadrant_filter else ''
    fig.suptitle(
        f'XAlign {quad_str}Top-{n} {direction}\n'
        f'(α={XALIGN_ALPHA}, β={XALIGN_BETA}, γ={XALIGN_GAMMA})',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


if len(df_with_xa) > 0:
    plot_xalign_topN(
        df_all, test_imgs, test_paths, test_masks,
        model, LAST_CONV_LAYER, BACKBONE_NAME,
        quadrant_filter='TP', sort_by='xalign', ascending=False, n=4,
        save_path=os.path.join(XALIGN_DIR, 'xalign_TP_best.png')
    )

    if 'FN' in df_all['quadrant'].values:
        plot_xalign_topN(
            df_all, test_imgs, test_paths, test_masks,
            model, LAST_CONV_LAYER, BACKBONE_NAME,
            quadrant_filter='FN', sort_by='xalign', ascending=True, n=4,
            save_path=os.path.join(XALIGN_DIR, 'xalign_FN_worst.png')
        )
    else:
        print('Tidak ada FN — semua tumor terdeteksi dengan benar.')

## 19. Metrik Statistik Klasifikasi (Sesuai Mahesh 2024, Section 3.5)

## 19. Metrik Statistik Klasifikasi (Mahesh 2024, Section 3.5)

In [ ]:
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f'Accuracy       : {acc:.4f}')
print(f'Precision      : {prec:.4f}')
print(f'Recall         : {rec:.4f}')
print(f'F1 Score       : {f1:.4f}')


### 19a. Classification Metrics

In [ ]:
cls_metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
cls_vals    = [acc, prec, rec, f1]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(cls_metrics, cls_vals,
              color=['#3498DB', '#9B59B6', '#E67E22', '#1ABC9C'],
              edgecolor='black', linewidth=0.8)
for bar, v in zip(bars, cls_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f'{v:.4f}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Metrics\n(Mahesh T R et al., 2024)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = os.path.join(XAI_DIR, 'metric_classification.png')
plt.savefig(out_path, dpi=150)
plt.show()

## 20. Distribusi XAlign Global & per Kuadran

## 20. Distribusi XAlign — Histogram

In [ ]:
if len(df_with_xa) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(df_with_xa['xalign'], bins=15,
            color='#1ABC9C', edgecolor='black', linewidth=0.7, alpha=0.85)
    ax.axvline(df_with_xa['xalign'].mean(), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {df_with_xa["xalign"].mean():.3f}')
    ax.set_xlabel('XAlign Score', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'Distribusi XAlign — Semua Gambar Tumor\n'
                  f'(α={XALIGN_ALPHA}, β={XALIGN_BETA}, γ={XALIGN_GAMMA})',
                  fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    out_path = os.path.join(XALIGN_DIR, 'xalign_histogram.png')
    plt.savefig(out_path, dpi=150)
    plt.show()
else:
    print('PERINGATAN: tidak ada data XAlign.')

## 20b. Distribusi XAlign — Boxplot per Kuadran

In [ ]:
if len(df_with_xa) > 0:
    colors_q = {'TP': '#27AE60', 'FN': '#C0392B', 'FP': '#E67E22', 'TN': '#2980B9'}
    quads    = [q for q in ['TP', 'FN', 'FP', 'TN'] if q in df_with_xa['quadrant'].values]
    data_box = [df_with_xa[df_with_xa['quadrant'] == q]['xalign'].values for q in quads]

    fig, ax = plt.subplots(figsize=(7, 5))
    bp = ax.boxplot(data_box, labels=quads, patch_artist=True, notch=False)
    for patch, q in zip(bp['boxes'], quads):
        patch.set_facecolor(colors_q.get(q, 'gray'))
        patch.set_alpha(0.75)
    ax.set_ylabel('XAlign Score', fontsize=12)
    ax.set_title(f'Distribusi XAlign per Kuadran (Boxplot)\n'
                  f'(α={XALIGN_ALPHA}, β={XALIGN_BETA}, γ={XALIGN_GAMMA})',
                  fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, linestyle='--', color='gray', alpha=0.4, label='0.5')
    ax.legend(fontsize=9)
    plt.tight_layout()
    out_path = os.path.join(XALIGN_DIR, 'xalign_boxplot.png')
    plt.savefig(out_path, dpi=150)
    plt.show()
else:
    print('PERINGATAN: tidak ada data XAlign.')

## 21. Sensitivity Analysis Bobot XAlign (Table 16, Muhammad 2025)

In [ ]:
if len(df_with_xa) > 0:
    weight_configs = [
        (0.6, 0.3, 0.1, '(0.6,0.3,0.1)'),
        (0.5, 0.4, 0.1, '(0.5,0.4,0.1) default'),
        (0.4, 0.4, 0.2, '(0.4,0.4,0.2)'),
        (0.3, 0.5, 0.2, '(0.3,0.5,0.2)'),
        (0.2, 0.6, 0.2, '(0.2,0.6,0.2)'),
    ]
    print(f'{"Config":<28} {"Mean":>10} {"SD":>10}')
    sens_data = []
    for alpha, beta, gamma, label in weight_configs:
        scores = [
            float(np.clip(alpha * r['wro'] + beta * r['bas'] - gamma * r['dp'], 0, 1))
            for r in df_with_xa.to_dict('records')
        ]
        m, s = np.mean(scores), np.std(scores)
        print(f'{label:<28} {m:>10.4f} {s:>10.4f}')
        sens_data.append({'config': label, 'mean': m, 'std': s})

    df_sens = pd.DataFrame(sens_data)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(df_sens['config'], df_sens['mean'], yerr=df_sens['std'],
           color='#3498DB', edgecolor='black', capsize=5, linewidth=0.8, alpha=0.85)
    for i, (m, s) in enumerate(zip(df_sens['mean'], df_sens['std'])):
        ax.text(i, m + s + 0.01, f'{m:.4f}', ha='center', fontsize=9)
    ax.set_ylabel('XAlign Mean Score', fontsize=12)
    ax.set_ylim(0, 1.2)
    ax.set_title('Sensitivity Analysis: XAlign vs Weight Configurations\n'
                  '(Muhammad & Bendechache 2025, Table 16)', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8, rotation=10)
    plt.tight_layout()
    out_path = os.path.join(XALIGN_DIR, 'xalign_sensitivity.png')
    plt.savefig(out_path, dpi=150)
    plt.show()

## 22. Simpan Ringkasan Lengkap (JSON)

In [ ]:
summary = {
    'model'           : 'EfficientNetB0 + Grad-CAM XAI',
    'reference_model' : 'Mahesh T R et al. (2024)',
    'reference_xalign': 'Muhammad D & Bendechache M (2025) — Front. Med. Technol. 7:1674343',
    'gradcam_layer'   : LAST_CONV_LAYER,
    'backbone'        : BACKBONE_NAME,
    'TP': len(tp_idx), 'TN': len(tn_idx), 'FP': len(fp_idx), 'FN': len(fn_idx),
    'accuracy'        : round(float(acc),  6),
    'precision'       : round(float(prec), 6),
    'recall'          : round(float(rec),  6),
    'f1_score'        : round(float(f1),   6),
    'xalign_enabled'  : MASKS_AVAILABLE,
    'xalign_alpha'    : XALIGN_ALPHA,
    'xalign_beta'     : XALIGN_BETA,
    'xalign_gamma'    : XALIGN_GAMMA,
    'xalign_n_images' : len(df_with_xa) if len(df_with_xa) > 0 else 0,
}

if len(df_with_xa) > 0:
    for col in ['xalign', 'wro', 'bas', 'dp']:
        summary[f'{col}_mean'] = round(float(df_with_xa[col].mean()), 6)
        summary[f'{col}_std' ] = round(float(df_with_xa[col].std()),  6)
    for q in ['TP', 'FN']:
        sub = df_with_xa[df_with_xa['quadrant'] == q]['xalign']
        if len(sub) > 0:
            summary[f'xalign_{q}_mean'] = round(float(sub.mean()), 6)
            summary[f'xalign_{q}_std' ] = round(float(sub.std()),  6)

summary_path = os.path.join(XAI_DIR, 'xai_xalign_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f'Output tersimpan di: {XAI_DIR}')
